# Section 6: LangChain, LlamaIndex & Agents — Hands-On

### GenAI Masterclass — From Frameworks to Autonomous Agents

---

In **Sections 1–5**, you built every piece of GenAI plumbing by hand:

- **Section 1** — How LLMs work (tokens, embeddings, attention)
- **Section 2** — Prompt engineering (zero-shot, few-shot, system prompts, JSON output)
- **Section 3** — From notebooks to apps (error handling, streaming, multi-provider)
- **Section 4** — RAG Part 1 (chunking, embeddings, ChromaDB, grounded prompts)
- **Section 5** — RAG Part 2 (hybrid search, re-ranking, evaluation)

Every helper function — `ask()`, `ask_json()`, the retry wrapper, the ChromaDB calls — you wrote yourself.

Now we **embrace frameworks**. The same patterns you wrote by hand are one-liners in LangChain and LlamaIndex. And on top of those primitives, we get something genuinely new: **agents** — LLMs that decide their own steps using tools.

This notebook walks through every concept from the Section 6 web page with running code.

---

## Setup

We'll use:

- **OpenAI** (`gpt-4o-mini`) — main reasoning model, matches Sections 1–5
- **Groq** (`llama-3.1-8b-instant`) — fast small model, matches your Best Practices notebook
- **Tavily** — real web search for the agent
- **TechStore docs** from Section 4 — consistent teaching data for the RAG chapters

Make sure your `.env` has `OPENAI_API_KEY`, `GROQ_API_KEY`, and `TAVILY_API_KEY`.

In [1]:
# Install required libraries (run once)
!pip install -q langchain langchain-openai langchain-groq langchain-community \
                langgraph tavily-python \
                llama-index llama-index-llms-openai llama-index-embeddings-openai \
                python-dotenv

You should consider upgrading via the '/Volumes/work/teach/genai-beginner/.venv/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify keys are loaded
for key in ["OPENAI_API_KEY", "GROQ_API_KEY", "TAVILY_API_KEY"]:
    status = "✅" if os.getenv(key) else "❌ MISSING"
    print(f"{status}  {key}")

In [ ]:
# ── Models we'll use throughout ──
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Main reasoning model (matches Sections 1-5)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Fast small model (matches Section 2 Best Practices notebook)
small_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

print("✅ Models ready:")
print(f"   llm       = {llm.model_name} (OpenAI)")
print(f"   small_llm = {small_llm.model_name} (Groq)")

---

## Chapter 3: LCEL — The Pipe Operator

### Continuing the Story...

LangChain components compose with Python's `|` operator, just like Unix shell commands. Read top-to-bottom: input → prompt → model → parser → output.

We'll start here because every LangChain example from now on uses LCEL syntax.

### 3.1 — Your First LCEL Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Three components
prompt = ChatPromptTemplate.from_template("Translate '{text}' to French. Return only the translation.")
parser = StrOutputParser()

# Compose with the pipe operator
chain = prompt | llm | parser

# Run it
result = chain.invoke({"text": "Hello, world"})
print(result)

That's a complete chain in three lines. Compare this to what you wrote in Section 2: build a messages list, call `client.chat.completions.create`, extract `response.choices[0].message.content`, handle errors. LCEL collapses all of it.

### 3.2 — Same Chain, Different Modes (Free with LCEL)

The real win: streaming, async, and batching come for free. Same `chain` object, different methods.

In [ ]:
# Mode 1: invoke (what we just did)
result = chain.invoke({"text": "Good morning"})
print(f"invoke:  {result}")

# Mode 2: batch — multiple inputs in parallel
results = chain.batch([
    {"text": "How are you?"},
    {"text": "See you tomorrow"},
    {"text": "Thank you very much"}
])
for r in results:
    print(f"batch:   {r}")

# Mode 3: stream — token-by-token output
print("stream:  ", end="", flush=True)
for chunk in chain.stream({"text": "I love programming"}):
    print(chunk, end="", flush=True)
print()

💡 **Key takeaway:** In Section 3 you wrote ~30 lines of streaming code by hand. LCEL gives it to you free, on the same chain object you already built.

### 3.3 — Swapping the Model (Provider Abstraction)

The chain doesn't care which provider sits in the middle. Swap `llm` for `small_llm` and the rest of the code is identical.

In [ ]:
# Same chain, but with the small Groq model instead of OpenAI
fast_chain = prompt | small_llm | parser

result = fast_chain.invoke({"text": "Where is the library?"})
print(f"Groq Llama 3.1 8B: {result}")

---

## Chapter 4: Chains in Practice

### Continuing the Story...

A chain is a pipeline. Three patterns come up over and over:

1. **Sequential refinement** — output of one step feeds the next
2. **Parallel fan-out** — multiple chains run on the same input
3. **Routing** — pick a chain based on the input

### 4.1 — Sequential: Draft → Critique → Revise

Generate a short pitch, critique it, then revise based on the critique. Output of each chain becomes input to the next.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

draft_prompt = ChatPromptTemplate.from_template(
    "Write a one-sentence elevator pitch for: {idea}"
)
critique_prompt = ChatPromptTemplate.from_template(
    "Critique this pitch in one sentence — what's weak about it?\n\nPitch: {draft}"
)
revise_prompt = ChatPromptTemplate.from_template(
    "Original pitch: {draft}\n"
    "Critique: {critique}\n\n"
    "Rewrite the pitch addressing the critique. One sentence only."
)

draft_chain = draft_prompt | llm | StrOutputParser()
critique_chain = critique_prompt | llm | StrOutputParser()
revise_chain = revise_prompt | llm | StrOutputParser()

# Compose sequentially. Each step's output becomes part of the next step's input.
full = (
    {"idea": RunnablePassthrough()}
    | RunnablePassthrough.assign(draft=draft_chain)
    | RunnablePassthrough.assign(critique=critique_chain)
    | revise_chain
)

result = full.invoke("a notebook app for GenAI engineers")
print(result)

### 4.2 — Parallel: Three Views in One Call

Run three chains in parallel on the same input. Total wall time ≈ slowest single chain (not sum of all three).

In [ ]:
from langchain_core.runnables import RunnableParallel
import time

pros_chain = ChatPromptTemplate.from_template(
    "List 3 pros of: {topic}. Be brief."
) | llm | StrOutputParser()

cons_chain = ChatPromptTemplate.from_template(
    "List 3 cons of: {topic}. Be brief."
) | llm | StrOutputParser()

summary_chain = ChatPromptTemplate.from_template(
    "Summarize {topic} in one sentence."
) | llm | StrOutputParser()

multi_view = RunnableParallel(
    pros=pros_chain,
    cons=cons_chain,
    summary=summary_chain
)

start = time.time()
result = multi_view.invoke({"topic": "remote work"})
elapsed = time.time() - start

print(f"⏱  Wall time: {elapsed:.1f}s (all 3 ran in parallel)\n")
print("─── PROS ───");    print(result["pros"])
print("\n─── CONS ───");  print(result["cons"])
print("\n─── SUMMARY ───"); print(result["summary"])

### 4.3 — Routing: Pick the Right Chain Based on Input

Use a classifier chain to decide which specialist chain runs next. The LLM is doing the routing decision.

In [ ]:
from langchain_core.runnables import RunnableLambda

# Three specialist chains
code_chain = ChatPromptTemplate.from_template(
    "You are a senior engineer. Answer this code question briefly: {question}"
) | llm | StrOutputParser()

math_chain = ChatPromptTemplate.from_template(
    "Solve this math problem step by step: {question}"
) | llm | StrOutputParser()

general_chain = ChatPromptTemplate.from_template(
    "Answer this question briefly: {question}"
) | llm | StrOutputParser()

# Classifier: returns one of: 'code' | 'math' | 'general'
classifier = ChatPromptTemplate.from_template(
    "Classify the question into ONE word — code, math, or general.\n\n"
    "Question: {question}\nClassification:"
) | llm | StrOutputParser()

def route(info):
    label = info["topic"].strip().lower()
    if "code" in label:
        return code_chain
    elif "math" in label:
        return math_chain
    return general_chain

router = (
    {"topic": classifier, "question": lambda x: x["question"]}
    | RunnableLambda(route)
)

# Try three different question types
for q in [
    "How do I reverse a string in Python?",
    "What is 23 * 47?",
    "What's the capital of France?"
]:
    print(f"❓ {q}")
    print(f"→  {router.invoke({'question': q})[:200]}")
    print()

💡 **Key takeaway:** Chains don't have to be linear. LCEL lets you compose sequential, parallel, and conditional flows using the same `|` syntax. The `RunnableParallel` and `RunnableLambda` primitives are the building blocks for everything more complex.

---

## Chapter 5: Memory

### Continuing the Story...

LLM calls are stateless — the model doesn't remember the previous turn. Memory is the abstraction for managing conversation history. Four types matter in practice:

| Type | Stores | Use when |
|------|--------|----------|
| Buffer | Full history verbatim | Short chats, plenty of token budget |
| Window | Last N turns | Long chats, only recent matters |
| Summary | LLM-generated running summary | Very long chats, lossy is OK |
| Vector | Embedded turns, retrieved by relevance | Old context might matter again |

### 5.1 — A Chat Without Memory (the problem)

Watch the model fail. Each call is independent — the second message has no idea who 'I' am.

In [ ]:
# No memory — just two separate LLM calls
print(llm.invoke("Hi, my name is Nij and I live in Dubai.").content)
print("---")
print(llm.invoke("What's my name and where do I live?").content)

The second answer is nonsense — the model can't see the first turn. Now let's add memory.

### 5.2 — RunnableWithMessageHistory (the modern way)

The recommended pattern in modern LangChain. We provide a function that returns a history store for a given session ID.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder

# Step 1: A simple chat prompt that has a slot for past messages
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Step 2: Plain chain
chain = chat_prompt | llm

# Step 3: Wrap with history. The store is just a dict mapping session_id → history.
store = {}
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Now both calls share the same session — the second remembers the first
config = {"configurable": {"session_id": "user-42"}}

print(chain_with_memory.invoke({"input": "Hi, my name is Nij and I live in Dubai."}, config).content)
print("---")
print(chain_with_memory.invoke({"input": "What's my name and where do I live?"}, config).content)

### 5.3 — Inspecting What Memory Actually Holds

The history isn't magic — it's a list of messages stored under each session ID. Let's look.

In [ ]:
# Peek at what's in memory for our session
history = get_history("user-42")
print(f"Messages stored: {len(history.messages)}\n")
for i, msg in enumerate(history.messages):
    print(f"[{i}] {type(msg).__name__:<12} {msg.content[:100]}")

### 5.4 — Window Memory: Keep Only the Last N Turns

For long chats, you don't want to send everything every time. Trim to the last N messages before passing to the model.

In [ ]:
from langchain_core.messages import trim_messages

# trim_messages keeps only the last N tokens (or messages) of history
trimmer = trim_messages(
    max_tokens=2,           # Keep only 2 messages worth
    strategy="last",
    token_counter=len,      # Use message count, not actual tokens
    include_system=True,
)

# Build a new chain that trims history before passing it in
windowed_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

windowed_chain = (
    RunnablePassthrough.assign(history=lambda x: trimmer.invoke(x["history"]))
    | windowed_prompt
    | llm
)

windowed_with_memory = RunnableWithMessageHistory(
    windowed_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Fresh session for this demo
config2 = {"configurable": {"session_id": "windowed-user"}}

# Build up some history
windowed_with_memory.invoke({"input": "I'm allergic to peanuts."}, config2)
windowed_with_memory.invoke({"input": "I live in Dubai."}, config2)
windowed_with_memory.invoke({"input": "I'm 35 years old."}, config2)

# Now ask about something from early in the chat — likely lost due to window
result = windowed_with_memory.invoke({"input": "What am I allergic to?"}, config2)
print(result.content)

💡 **Key takeaway:** Memory is just a list of messages stored under a session ID and threaded into every prompt. Different memory types are different rules for *which* messages to include. In production, you back the store with Redis/Postgres instead of `InMemoryChatMessageHistory`.

---

## Chapter 6: Tools

### Continuing the Story...

A tool is any function the LLM can call. The model doesn't *execute* the function — it produces a structured request saying "I want to call X with these arguments." Your code runs the function and feeds the result back.

### 6.1 — Defining a Tool with @tool

In [ ]:
from langchain_core.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a ticker symbol like AAPL or TSLA."""
    # In real life: call a market data API
    prices = {"AAPL": 224.50, "TSLA": 248.10, "GOOGL": 178.30, "MSFT": 415.20}
    if ticker.upper() in prices:
        return f"{ticker.upper()} is trading at ${prices[ticker.upper()]}"
    return f"No price found for {ticker}"

# Inspect what the LLM will see
print(f"Tool name:        {get_stock_price.name}")
print(f"Tool description: {get_stock_price.description}")
print(f"Tool args schema: {get_stock_price.args}")

The docstring becomes the description the LLM reads. **This is what the LLM uses to decide whether to call the tool.** Write it carefully.

### 6.2 — Binding Tools to a Model

In [ ]:
# Bind the tool — now the model knows it can call get_stock_price
llm_with_tools = llm.bind_tools([get_stock_price])

response = llm_with_tools.invoke("What's Apple trading at right now?")

print("Content:", response.content or "(empty — model chose to call a tool instead)")
print("\nTool calls:")
for tc in response.tool_calls:
    print(f"  → {tc['name']}({tc['args']})")

Notice the model didn't *answer* — it produced a tool call. The model never executes anything. We do.

### 6.3 — The Manual Tool-Calling Loop

Before we let frameworks hide it, let's run the loop ourselves so you see exactly what's happening.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# Step 1: Send the user's question with tools bound
messages = [HumanMessage("What's Apple trading at, and how about Tesla?")]
response = llm_with_tools.invoke(messages)
messages.append(response)

print(f"Step 1 — model wants to call {len(response.tool_calls)} tool(s)")
for tc in response.tool_calls:
    print(f"   → {tc['name']}({tc['args']})")

# Step 2: Execute each tool call and append results as ToolMessages
for tc in response.tool_calls:
    result = get_stock_price.invoke(tc["args"])
    messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))
    print(f"\nStep 2 — ran {tc['name']}: {result}")

# Step 3: Send results back to the model so it can produce the final answer
final = llm_with_tools.invoke(messages)
print(f"\nStep 3 — final answer:\n{final.content}")

💡 **Key takeaway:** That loop — call model → run tools → send results back → call model again — is what an *agent* is. An agent framework just runs this loop for you, with safety limits, retries, and tracing baked in.

---

## Chapter 7: Callbacks & LangSmith

### Continuing the Story...

LangSmith is LangChain's hosted observability platform. With two environment variables, every chain run shows up in a UI with full inputs, outputs, latency, cost, and intermediate steps.

**You don't need a paid plan to follow along** — LangSmith has a free tier that covers personal use comfortably.

### 7.1 — Enabling Tracing

Set these in your `.env`:

```
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__your_key_here
LANGCHAIN_PROJECT=genai-decoded-section-6
```

Once enabled, **every** `chain.invoke()` from this notebook gets logged to LangSmith automatically — no other code changes needed.

In [ ]:
# Check whether tracing is on
tracing_on = os.getenv("LANGCHAIN_TRACING_V2") == "true"
project = os.getenv("LANGCHAIN_PROJECT", "(not set)")

if tracing_on:
    print(f"✅ LangSmith tracing is ON")
    print(f"   Project: {project}")
    print(f"   Visit https://smith.langchain.com to see your traces")
else:
    print("ℹ️  LangSmith tracing is OFF.")
    print("   Set LANGCHAIN_TRACING_V2=true and LANGCHAIN_API_KEY in .env to enable.")
    print("   Then re-run any chain in this notebook to see it appear in LangSmith.")

💡 **Key takeaway:** Production rule — turn on tracing from day one. The cost of *not* having traces when something breaks in production is far higher than the (very small) cost of running them.

---

## Chapter 8: LlamaIndex Data Connectors

### Switching Frameworks: Welcome to LlamaIndex

LangChain shines at orchestration. LlamaIndex shines at **getting your data in**. The LlamaHub registry has 300+ connectors, all returning the same `Document` shape.

We'll use the TechStore docs from Section 4 as our consistent teaching data.

### 8.1 — Setting Up Sample Documents

Reusing TechStore docs from Section 4. If they're not present in the expected location, we'll create minimal versions inline so the notebook is self-contained.

In [ ]:
from pathlib import Path

# Path used in Section 4
techstore_dir = Path("../4_rag_part1/sample_docs")
local_dir = Path("./sample_docs")

if techstore_dir.exists():
    docs_dir = techstore_dir
    print(f"✅ Using TechStore docs from Section 4: {docs_dir.resolve()}")
else:
    # Fallback: create minimal copies so the notebook still works standalone
    local_dir.mkdir(exist_ok=True)
    (local_dir / "return_policy.txt").write_text(
        "TechStore Return Policy\n\n"
        "Customers may return any product within 30 days of purchase for a full refund. "
        "Returns must include the original packaging and proof of purchase. "
        "Opened software is non-refundable. "
        "Damaged items can be exchanged but not refunded."
    )
    (local_dir / "employee_handbook.txt").write_text(
        "TechStore Employee Handbook\n\n"
        "All employees receive 20 days of paid leave per year. "
        "Remote work is permitted up to 2 days per week with manager approval. "
        "Performance reviews happen quarterly. "
        "Health insurance includes dental and vision coverage."
    )
    (local_dir / "product_catalog.txt").write_text(
        "TechStore Product Catalog\n\n"
        "Laptops: ProBook 15 ($1,299), UltraSlim 13 ($999), GamingMax 17 ($1,899). "
        "All laptops come with a 2-year warranty. "
        "Accessories: wireless mouse ($29), USB-C hub ($49), laptop stand ($79)."
    )
    docs_dir = local_dir
    print(f"ℹ️  TechStore docs not found — created fallback in: {docs_dir.resolve()}")

print(f"\nFiles available:")
for f in sorted(docs_dir.iterdir()):
    print(f"   {f.name}")

### 8.2 — SimpleDirectoryReader: One Line, Many Formats

`SimpleDirectoryReader` handles `.txt`, `.pdf`, `.docx`, `.md`, `.csv`, images, and more — automatically picking the right parser per extension.

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(str(docs_dir)).load_data()

print(f"Loaded {len(documents)} documents\n")
for doc in documents:
    print(f"📄 {doc.metadata.get('file_name', 'unknown')}")
    print(f"   Length: {len(doc.text)} chars")
    print(f"   Preview: {doc.text[:120]}...")
    print()

### 8.3 — Custom Connector: Web Page

Different connector, same `Document` output. The downstream pipeline doesn't care where data came from.

In [ ]:
# Note: only run this cell if you have internet access
from llama_index.readers.web import SimpleWebPageReader

# Same uniform Document interface, different source
web_docs = SimpleWebPageReader().load_data([
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
])

print(f"Loaded {len(web_docs)} web document(s)")
print(f"First 200 chars:\n{web_docs[0].text[:200]}...")

💡 **Key takeaway:** All 300+ LlamaIndex connectors return the same `Document` shape. Load once with the right connector — the rest of your pipeline (chunking, indexing, querying) stays identical regardless of source.

---

## Chapter 9: Indices — Beyond Just Vector Search

### Continuing the Story...

In Sections 4–5, "the index" meant a vector store. LlamaIndex generalizes: an **index** is any data structure built over your documents to make retrieval efficient.

We'll demo the two most common types — `VectorStoreIndex` (the default) and `SummaryIndex` (for exhaustive coverage).

### 9.1 — VectorStoreIndex: Three Lines to Working RAG

In [ ]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.openai import OpenAI as LIOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LlamaIndex to use the same models as our LangChain code
Settings.llm = LIOpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# The whole RAG pipeline in three lines
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
response = query_engine.query("What's our return policy?")

print(response)

Compare this to your Section 4 code: you wrote ~80 lines for chunking, embedding, ChromaDB setup, retrieval, and prompt assembly. LlamaIndex does it in three lines with sensible defaults.

### 9.2 — Inspecting What Was Retrieved

The response includes the source nodes that grounded the answer.

In [ ]:
# Same query, but inspect the retrieved chunks
response = query_engine.query("Can I return opened software?")

print(f"Answer: {response}\n")
print(f"Retrieved {len(response.source_nodes)} chunks:\n")
for i, node in enumerate(response.source_nodes):
    print(f"[{i}] Score: {node.score:.3f}")
    print(f"    Source: {node.metadata.get('file_name', 'unknown')}")
    print(f"    Text: {node.text[:150]}...")
    print()

### 9.3 — SummaryIndex: Read Everything, Miss Nothing

When the question is "summarize all of X" rather than "find specific Y", `SummaryIndex` reads through every chunk instead of relying on top-K retrieval.

In [ ]:
from llama_index.core import SummaryIndex

summary_index = SummaryIndex.from_documents(documents)
summary_engine = summary_index.as_query_engine()

# A question where exhaustive coverage matters
response = summary_engine.query(
    "Summarize all the policies and benefits TechStore offers, across all documents."
)
print(response)

💡 **Key takeaway:** `VectorStoreIndex` for "find the relevant bit" — fast, top-K. `SummaryIndex` for "read everything and tell me" — slower but exhaustive. Pick the index that matches the question shape.

---

## Chapter 10: Query Engines & Routers

### Continuing the Story...

A `RouterQueryEngine` uses an LLM to pick *which* index to query based on the question. This is how production apps with multiple data domains work.

### 10.1 — Building a Router Over Two Indices

We'll build separate indices for the return policy and the employee handbook, then let a router LLM decide which to use per question.

In [ ]:
# Build two specialist indices, one per document
return_doc = [d for d in documents if "return" in d.metadata.get("file_name", "").lower()]
hr_doc     = [d for d in documents if "employee" in d.metadata.get("file_name", "").lower()]

return_index = VectorStoreIndex.from_documents(return_doc) if return_doc else None
hr_index     = VectorStoreIndex.from_documents(hr_doc)     if hr_doc else None

print(f"return_index built: {return_index is not None}")
print(f"hr_index built:     {hr_index is not None}")

In [ ]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# Wrap each index as a tool with a clear description
# (the description is what the router LLM reads to choose)
tools = []
if return_index:
    tools.append(QueryEngineTool.from_defaults(
        query_engine=return_index.as_query_engine(),
        description="Use for questions about returns, refunds, exchanges, and product purchase policies."
    ))
if hr_index:
    tools.append(QueryEngineTool.from_defaults(
        query_engine=hr_index.as_query_engine(),
        description="Use for questions about employee benefits, leave, remote work, and HR policies."
    ))

router = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=tools,
    verbose=True,  # Show which tool the router picks
)

# Two questions, each routes to a different tool
print("\n=== Question 1 ===")
print(router.query("How many vacation days do employees get?"))

print("\n=== Question 2 ===")
print(router.query("Can I return opened software?"))

💡 **Key takeaway:** A router is an LLM picking between options based on descriptions — the same primitive that powers full agents in the next chapter. Routers are agents with one decision; agents are routers with many.

---

## Chapter 12: The ReAct Loop

### Welcome to Agents

We've reached the centerpiece. An **agent** is an LLM in a loop with tools, deciding its own steps. The dominant pattern is **ReAct** — at each step the model produces a *Thought*, an *Action* (tool call), and processes the *Observation* (tool result), until it decides it's done.

### 12.1 — A Tiny Agent with One Tool

Let's start small — one tool, one query, full visibility into what the agent is doing.

In [ ]:
from langgraph.prebuilt import create_react_agent

# Reuse our stock price tool from Chapter 6
agent = create_react_agent(llm, [get_stock_price])

result = agent.invoke({
    "messages": [("user", "What's Apple trading at?")]
})

# Print every message in the trace so we see the loop
for msg in result["messages"]:
    msg_type = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"[{msg_type}] tool_calls = {[tc['name'] for tc in msg.tool_calls]}")
    else:
        content = msg.content[:200] if msg.content else "(empty)"
        print(f"[{msg_type}] {content}")
    print()

You should see four messages: HumanMessage (user query) → AIMessage with tool_calls → ToolMessage (result) → AIMessage (final answer). That's one ReAct iteration.

### 12.2 — Watching a Multi-Step Trace

Now a question that needs two tool calls. Watch the agent chain them.

In [ ]:
# Add a second tool: percent change calculator
@tool
def percent_change(start: float, end: float) -> str:
    """Calculate the percent change from start to end. Returns a string like '+12.3%'."""
    pct = (end - start) / start * 100
    sign = "+" if pct >= 0 else ""
    return f"{sign}{pct:.2f}%"

agent_v2 = create_react_agent(llm, [get_stock_price, percent_change])

result = agent_v2.invoke({
    "messages": [("user",
        "Apple was at $180 a year ago. What's the percent change to today's price?")]
})

# Pretty-print the trace
for i, msg in enumerate(result["messages"]):
    label = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{i}] {label}  →  {tc['name']}({tc['args']})")
    elif label == "ToolMessage":
        print(f"[{i}] {label}  ←  {msg.content}")
    else:
        content = msg.content[:200] if msg.content else "(empty)"
        print(f"[{i}] {label}: {content}")

💡 **Key takeaway:** The agent figured out it needed `get_stock_price` first (for today's price), *then* `percent_change` (using last year's price + today's). You wrote zero orchestration code — the LLM ordered the calls.

---

## Chapter 13: Building a Real Agent

### The Capstone

Three tools — web search (real, via Tavily), calculator, exchange rates — wired into one agent. Then we ask a question that needs all three.

### 13.1 — Defining the Three Tools

In [ ]:
from tavily import TavilyClient
import requests

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(query: str) -> str:
    """Search the web for current events, news, prices, sports scores, or any
    information that may have changed recently. Use this for anything where
    fresh real-world data matters. Do NOT use for general knowledge or pure math."""
    results = tavily.search(query, max_results=3)
    snippets = [r["content"] for r in results.get("results", [])]
    return "\n\n".join(snippets) if snippets else "No results."

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression in Python syntax.
    Example: calculate('(224.50 - 180.30) / 180.30 * 100')"""
    try:
        # Sandboxed eval — no builtins, no globals
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    """Get the current exchange rate between two currencies.
    Currencies are 3-letter codes like USD, EUR, AED, INR."""
    r = requests.get(f"https://api.exchangerate-api.com/v4/latest/{from_currency.upper()}")
    rates = r.json().get("rates", {})
    rate = rates.get(to_currency.upper())
    if rate is None:
        return f"Could not find rate for {from_currency} → {to_currency}"
    return f"1 {from_currency.upper()} = {rate} {to_currency.upper()}"

print("✅ Three tools defined")
for t in [web_search, calculate, get_exchange_rate]:
    print(f"   • {t.name}: {t.description.split(chr(10))[0]}")

### 13.2 — Wiring the Agent and Running It

One `create_react_agent` call. The LangGraph runtime handles the loop, retries, parallel calls, and message threading.

In [ ]:
tools = [web_search, calculate, get_exchange_rate]
real_agent = create_react_agent(llm, tools)

question = (
    "What's Apple's current stock price in AED, "
    "and how much higher (in dollars) is it than $200?"
)

result = real_agent.invoke({"messages": [("user", question)]})

# Print the full trace
print("=" * 70)
print(f"USER: {question}")
print("=" * 70)
for msg in result["messages"][1:]:  # skip the user message we already printed
    label = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            args_short = str(tc['args'])[:120]
            print(f"\n→ TOOL CALL: {tc['name']}({args_short})")
    elif label == "ToolMessage":
        content = msg.content[:300]
        print(f"← TOOL RESULT: {content}")
    elif msg.content:
        print(f"\n🤖 FINAL ANSWER:\n{msg.content}")

### 13.3 — Streaming the Agent's Reasoning

For real apps, you want to see the agent think in real time. `astream_events` gives you a token-by-token stream of every step.

In [ ]:
# Stream version — see steps as they happen
print("Streaming agent steps:\n")

for chunk in real_agent.stream(
    {"messages": [("user", "What's the current price of Tesla in EUR?")]},
    stream_mode="values"
):
    last_msg = chunk["messages"][-1]
    label = type(last_msg).__name__
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"→  {tc['name']}({tc['args']})")
    elif label == "ToolMessage":
        print(f"←  {last_msg.content[:200]}")
    elif last_msg.content and label == "AIMessage":
        print(f"🤖  {last_msg.content[:300]}")

### 13.4 — Setting Safety Limits

Real agents need guardrails. The two most important: max iterations (prevents infinite loops) and recursion limit (prevents runaway recursion).

In [ ]:
from langgraph.errors import GraphRecursionError

# Configure with a tight recursion limit
try:
    result = real_agent.invoke(
        {"messages": [("user", "What's the price of Microsoft stock?")]},
        config={"recursion_limit": 10}  # Hard ceiling — stops if exceeded
    )
    print("✅ Completed within limits")
    print(result["messages"][-1].content[:300])
except GraphRecursionError as e:
    print(f"❌ Hit recursion limit: {e}")

💡 **Key takeaway:** You wrote three small functions and one `create_react_agent` line. The LLM picked the tools, ordered the calls, threaded the results, and produced the final answer. That's the leverage frameworks give you — and why we spent five sections building primitives by hand first, so you'd appreciate it.

---

## Wrap-Up: What You've Built

In this notebook you have running examples of:

- **LCEL pipelines** — `prompt | llm | parser` with free streaming, batching, async
- **Composition patterns** — sequential, parallel, conditional routing
- **Memory** — `RunnableWithMessageHistory` with sessions and trimming
- **Tools** — `@tool` decorator, the manual tool-calling loop, why it's the agent loop in disguise
- **LangSmith** — production tracing setup (one env var)
- **LlamaIndex** — `SimpleDirectoryReader`, `VectorStoreIndex`, `SummaryIndex`, router engines
- **Agents** — single-tool, multi-step, and a three-tool capstone with real web search

### Practice Assignments (from the web page)

1. **LangChain Agent** — Add a fourth tool of your own to the Chapter 13 agent. Run it on a question that uses all four. Submit a LangSmith trace screenshot.
2. **LlamaIndex Query Engine** — Add a third index (e.g., product catalog) and extend the router. Compare default vs. `tree_summarize` modes.
3. **Same RAG, Both Frameworks** — Re-build the Chapter 9 RAG pipeline in LangChain (using `Chroma` + `RetrievalQA`-style chains). Write up which felt easier and where each abstraction helped.
4. **LangSmith Exploration** — Find the most expensive call in your traces. Identify one prompt or tool description you could improve.

---

**Next up: Section 7** — building on agents with multi-agent systems, planning, and long-horizon tasks.